# Credit Default Prediction — Data Ingestion & Preprocessing

## Course: AIML ZG549 — API Driven Cloud Native Solutions
### Work Integrated Learning Programmes Division, BITS Pilani

**Student Details & Group Information**
* **Group Number:** Group 82
* **Student Names & IDs:**
  * Student 1 Palash Raj (ID: 2025AE05803)
  * Student 2 NIKHIL (ID: 2025ae05760)
  * Student 3 TEJAS (ID: 2025AE05838)
  * Student 4 AMRUTA (ID: 2025AE05435)

## The business problem
"The business objective is to predict the probability that a customer will experience financial distress in the next two years, allowing financial institutions to optimize credit approval workflows and mitigate default risks."

In [19]:
import os
import pandas as pd
import kagglehub
import logging
from sklearn.preprocessing import StandardScaler

# ==========================================
# Task 1.1: Environment Setup
# ==========================================
def setup_directories():
    """Create the required project repository with structured directories."""
    directories = ['./data', './src/preprocessing', './logs']
    for d in directories:
        os.makedirs(d, exist_ok=True)

def ingest_and_preprocess():
    # Setup directories before configuring file-based logging
    setup_directories()
    
    # Configure logging to write to the new /logs directory
    logging.basicConfig(
        level=logging.INFO, 
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler("./logs/preprocessing.log"),
            logging.StreamHandler()
        ]
    )
    
    logging.info("Starting data ingestion from Kaggle...")
    
    # ==========================================
    # Task 1.1: Data Ingestion
    # ==========================================
    path = kagglehub.dataset_download("brycecf/give-me-some-credit-dataset")
    file_path = os.path.join(path, "cs-training.csv")
    df = pd.read_csv(file_path)
    
    initial_rows, initial_cols = df.shape
    logging.info(f"Dataset loaded successfully. Shape: {initial_rows} rows, {initial_cols} columns.")
    
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
        logging.info("Removed 'Unnamed: 0' column.")
        
    # ==========================================
    # Task 1.2: Summary Statistics
    # ==========================================
    numeric_cols = ['DebtRatio', 'MonthlyIncome', 'RevolvingUtilizationOfUnsecuredLines']
    stats_df = pd.DataFrame({
        'Mean': df[numeric_cols].mean(),
        'Median': df[numeric_cols].median(),
        'Std_Dev': df[numeric_cols].std(),
        'Skewness': df[numeric_cols].skew()
    })
    logging.info(f"Summary Statistics calculated:\n{stats_df}")
    
    # ==========================================
    # Task 1.2: Missing Data Handling
    # ==========================================
    missing_income = df['MonthlyIncome'].isnull().sum()
    missing_dependents = df['NumberOfDependents'].isnull().sum()
    total_imputed = missing_income + missing_dependents
    
    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())
    
    # ==========================================
    # Task 1.3: Data Type Verification 
    # ==========================================
    df['NumberOfDependents'] = df['NumberOfDependents'].astype(int)
    df['SeriousDlqin2yrs'] = df['SeriousDlqin2yrs'].astype(int)
    logging.info(f"Imputed {total_imputed} missing values and verified data types (Target label converted to integer).")
    
    # ==========================================
    # Task 1.3: Normalization
    # ==========================================
    # We must not scale the target column ('SeriousDlqin2yrs')
    features_to_scale = df.columns.drop('SeriousDlqin2yrs')
    scaler = StandardScaler()
    
    # Apply StandardScaler
    df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
    logging.info("Applied StandardScaler to numerical features.")
    
    # Save the processed dataset for the EDA stage
    processed_file_path = './data/processed_credit_data.csv'
    df.to_csv(processed_file_path, index=False)
    logging.info(f"Processed dataset saved successfully to {processed_file_path}")

    # Generate Metadata for API (Sub-Objective 2)
    preprocessing_metadata = {
        "pipeline_step": "preprocessing",
        "initial_records": int(initial_rows),
        "missing_values_imputed": int(total_imputed),
        "imputation_strategy": "median",
        "status": "success"
    }
    
    logging.info("Data ingestion and preprocessing step finished successfully.")
    
    return df, preprocessing_metadata

# For local testing of this block:
if __name__ == "__main__":
    processed_df, meta = ingest_and_preprocess()

# Conclusion

The data ingestion and preprocessing stage has been completed successfully.

### Key outcomes

1. The Kaggle credit dataset was downloaded and loaded into Pandas.
2. The dataset initially contained **150,000 records** and **12 columns**.
3. The unnecessary `Unnamed: 0` column was removed, leaving **11 useful columns**.
4. Summary statistics, including mean, median, standard deviation, and skewness, were calculated.
5. Missing values were identified in `MonthlyIncome` and `NumberOfDependents`.
6. Missing values were imputed using the median:
   - `MonthlyIncome`: **5,400**
   - `NumberOfDependents`: **0**
7. After imputation, all columns contained **zero missing values**.
8. Data types were verified and `NumberOfDependents` was converted to integer.
9. Numerical features were normalized using `StandardScaler`.
10. The final processed dataset was saved as **`processed_credit_data.csv`**.

The processed dataset is now ready to be used by (Member 2).

Code for Member 2: Data Scientist / EDA Lead

In [20]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import logging

def perform_eda(df):
    logging.info("Starting EDA phase...")
    
    # ==========================================
    # Task 2.1: Correlation Analysis
    # ==========================================
    plt.figure(figsize=(12, 8))
    corr_matrix = df.corr(method='pearson')
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
    plt.title('Pearson Correlation Matrix')
    plt.tight_layout()
    plt.savefig('correlation_matrix.png')  # Saves to disk instead of displaying
    plt.close()
    logging.info("Correlation matrix saved as 'correlation_matrix.png'.")
    
    # ==========================================
    # Task 2.2: Binning & One-Hot Encoding
    # ==========================================
    df_engineered = df.copy()
    
    # Binning 'age'
    age_bins = [0, 25, 45, 65, 120]
    age_labels = ['Under_25', '25_to_45', '45_to_65', 'Over_65']
    df_engineered['AgeGroup'] = pd.cut(df_engineered['age'], bins=age_bins, labels=age_labels, right=False)
    
    # Binning 'MonthlyIncome'
    income_bins = [0, 3000, 5000, 8000, 15000, np.inf]
    income_labels = ['Low', 'Lower_Middle', 'Middle', 'Upper_Middle', 'High']
    df_engineered['IncomeGroup'] = pd.cut(df_engineered['MonthlyIncome'], bins=income_bins, labels=income_labels, right=False)
    
    # Encoding
    df_encoded = pd.get_dummies(df_engineered, columns=['AgeGroup', 'IncomeGroup'], drop_first=False)
    logging.info(f"Binning and encoding complete. New shape: {df_encoded.shape}")
    
    # ==========================================
    # Task 2.3: Feature Importance
    # ==========================================
    target_col = 'SeriousDlqin2yrs'
    X = df_encoded.drop(columns=[target_col])
    y = df_encoded[target_col]
    
    rf_model = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=10)
    rf_model.fit(X, y)
    
    feat_imp_df = pd.DataFrame({
        'Feature': X.columns, 
        'Importance': rf_model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis')
    plt.title('Random Forest Feature Importance (Gini)')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    logging.info("Feature importance plot saved as 'feature_importance.png'.")
    
    # ==========================================
    # Task 2.4: Univariate & Bivariate Visualizations
    # ==========================================
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Univariate: Debt Ratio
    sns.histplot(df['DebtRatio'], bins=50, ax=axes[0], color='blue', kde=False)
    axes[0].set_yscale('log')
    axes[0].set_title('Distribution of DebtRatio (Log Scale)')
    
    # 2. Univariate: Revolving Utilization Boxplot
    sns.boxplot(x=df['RevolvingUtilizationOfUnsecuredLines'], ax=axes[1], color='orange')
    axes[1].set_xscale('log')
    axes[1].set_title('Boxplot: Revolving Utilization')
    
    # 3. Bivariate: Default Rate by Age
    default_rates = df_engineered.groupby('AgeGroup', observed=True)['SeriousDlqin2yrs'].mean().reset_index()
    default_rates['SeriousDlqin2yrs'] *= 100 
    sns.barplot(x='AgeGroup', y='SeriousDlqin2yrs', data=default_rates, palette='magma', ax=axes[2])
    axes[2].set_title('Default Rate (%) by Age Bracket')
    
    plt.tight_layout()
    plt.savefig('eda_plots.png')
    plt.close()
    logging.info("Univariate and Bivariate plots saved as 'eda_plots.png'.")
    
    # ==========================================
    # API Metadata Generation
    # ==========================================
    top_feature_name = str(feat_imp_df.iloc[0]['Feature'])
    top_feature_score = float(feat_imp_df.iloc[0]['Importance'])
    
    eda_metadata = {
        "pipeline_step": "exploratory_data_analysis",
        "top_predictive_feature": top_feature_name,
        "top_feature_importance_score": round(top_feature_score, 4),
        "visualizations_generated": 3,
        "status": "success"
    }
    
    logging.info("EDA pipeline step finished successfully.")
    return df_encoded, eda_metadata

# Conclusion for Member 2

The Exploratory Data Analysis (EDA) phase has been successfully completed.

### Key Outcomes
1. **Correlation Analysis:** A Pearson correlation matrix was generated and saved as `correlation_matrix.png` to identify multicollinearity among numeric features.
2. **Feature Engineering:** Continuous variables like `Age` and `MonthlyIncome` were successfully binned into categorical risk groups and One-Hot Encoded for model readiness.
3. **Feature Importance:** A Random Forest Classifier successfully evaluated the drivers of default risk. The Gini-importance plot was saved as `feature_importance.png`, highlighting the most predictive financial behaviors.
4. **Visualizations:** Bivariate and univariate plots (saved as `eda_plots.png`) were created to compare default rates across different age brackets and analyze the distribution of debt ratios and revolving utilization. 

The engineered dataset is now ready for automated pipeline integration by Member 3.

# Member 3: Cloud & DataOps Engineer (Pipeline Automation)

This section focuses on automating the data pipeline, scheduling it, and implementing centralized logging. 

### Objectives
- Encapsulate the functions from Member 1 and Member 2 into an automated pipeline script.
- Implement centralized logging to track timestamp, records processed, missing values imputed, and execution status[cite: 1].
- Schedule the pipeline to run every 2 minutes using `APScheduler`[cite: 1, 2].
- Output operational metrics to a structured JSON file simulating a centralized cloud logging system (ready for AWS CloudWatch/GCP Logging or Grafana)[cite: 1].

In [ ]:
import time
import json
import logging
from datetime import datetime
import os

# ==========================================\n
# Task 3.2: Centralized Logging Setup
# ==========================================\n
# Setting up a structured JSON logger to simulate streaming to cloud systems (AWS CloudWatch/GCP/ELK)
LOG_FILE = 'pipeline_metrics.json'

def log_pipeline_metrics(metrics_data):
    """Appends structured JSON logs to a file for dashboard and API integration."""
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "level": "INFO",
        "metrics": metrics_data
    }
    
    # Write to local JSON log file for Member 4's API and Dashboard integration
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(log_entry) + '\n')
    
    # Standard console logging
    logging.info(f"PIPELINE METRICS LOGGED: {json.dumps(metrics_data)}")

# ==========================================\n
# Task 3.1: Pipeline Scripting & Encapsulation
# ==========================================\n
def run_pipeline():
    """Encapsulates Member 1 and Member 2 modules into an automated execution flow."""
    logging.info("--- STARTING AUTOMATED PIPELINE RUN ---")
    start_time = time.time()
    
    try:
        # Step 1: Data Ingestion & Preprocessing (Member 1)
        df, prep_metadata = ingest_and_preprocess()
        
        # Step 2: Exploratory Data Analysis (Member 2)
        df_encoded, eda_metadata = perform_eda(df)
        
        execution_time = round(time.time() - start_time, 2)
        
        # Compile comprehensive operational metrics
        pipeline_metrics = {
            "execution_status": "success",
            "execution_time_seconds": execution_time,
            "records_processed": prep_metadata["initial_records"],
            "missing_values_imputed": prep_metadata["missing_values_imputed"],
            "top_feature_driver": eda_metadata["top_predictive_feature"],
            "top_feature_score": eda_metadata["top_feature_importance_score"]
        }
        
        # Stream metrics to centralized logs
        log_pipeline_metrics(pipeline_metrics)
        logging.info("--- PIPELINE RUN COMPLETED SUCCESSFULLY ---")
        
        return pipeline_metrics
        
    except Exception as e:
        # Log failure metrics
        execution_time = round(time.time() - start_time, 2)
        error_metrics = {
            "execution_status": "failed",
            "execution_time_seconds": execution_time,
            "error_message": str(e)
        }
        log_pipeline_metrics(error_metrics)
        logging.error(f"--- PIPELINE RUN FAILED: {str(e)} ---")
        return error_metrics

# Run it once manually to test the flow
test_run_metrics = run_pipeline()

### Task 3.1 & 3.3: Scheduling and Dashboard Readiness
To fulfill the requirement of running the pipeline every 2 minutes[cite: 1, 2], we utilize `APScheduler`. This allows the notebook (or a standalone `.py` script) to run the task asynchronously in the background.

The structured logs outputted to `pipeline_metrics.json` act as the backend data source for the Cloud Dashboard (Task 3.3) and the built-in API endpoints that Member 4 will construct[cite: 1].

In [ ]:
# Install apscheduler if not present: !pip install apscheduler
from apscheduler.schedulers.background import BackgroundScheduler
import time

# Initialize the scheduler
scheduler = BackgroundScheduler()

# Add the pipeline job to run every 2 minutes
scheduler.add_job(
    func=run_pipeline, 
    trigger="interval", 
    minutes=2,
    id='credit_default_pipeline_job',
    name='Run Data Ingestion and EDA Pipeline every 2 mins',
    replace_existing=True
)

# Start the scheduler
scheduler.start()
logging.info("Scheduler started. Pipeline will execute every 2 minutes. Press Interrupt/Stop in Jupyter to halt.")

# Note: In a real Cloud environment, this could also be triggered by AWS EventBridge or GCP Cloud Scheduler.
# For the Jupyter Notebook environment, APScheduler runs it continuously in the background.

In [ ]:
# Run this cell ONLY when you want to stop the automated 2-minute executions.
# scheduler.shutdown()
# logging.info("Scheduler stopped.")

# Conclusion for Member 3

The data pipeline automation and centralized logging setup have been successfully implemented.

### Key Outcomes
1. **Pipeline Encapsulation:** The preprocessing (Member 1) and EDA (Member 2) tasks were combined into a single unified execution flow.
2. **Task Scheduling:** `BackgroundScheduler` was configured to automatically trigger the pipeline every 2 minutes.
3. **Centralized Logging:** Operational metrics (execution time, records processed, imputation counts, and status) are now continuously streamed to a structured file (`pipeline_metrics.json`).

### Task 3.3: Cloud Dashboard Integration
*Note: A real-time dashboard cannot be hosted natively within a Jupyter Notebook cell.* 

To fulfill the cloud dashboard requirement, a separate Streamlit application (`dashboard.py`) must be used. It reads the `pipeline_metrics.json` file to visualize the pipeline's health in real-time. 

**To run the dashboard:**
1. Ensure `streamlit` is installed (`pip install streamlit`).
2. Run the external script via terminal: `streamlit run dashboard.py`
3. The dashboard will automatically refresh to show the 2-minute interval updates.

# Member 4: Backend / API Engineer (API & Documentation)

This section covers the final API layer construction, endpoint testing, and the assembly of the final project deliverables.

### Task 4.1: Built-in API Endpoints (FastAPI)
The following FastAPI implementation exposes core application metrics and reads directly from the centralized logs generated by Member 3's pipeline script. 

**Execution Note:** 
Because a web server cannot be continuously hosted in a standard Jupyter Notebook cell, you must save the Python code below into a file named `api_app.py`. 
Then, run it from your terminal using the following command:
`uvicorn api_app:app --reload --port 8000`

In [ ]:
from fastapi import FastAPI, HTTPException
import json
import os

app = FastAPI(
    title="Credit Default Pipeline API",
    description="API to access data pipeline metrics and EDA details.",
    version="1.0.0"
)

LOG_FILE = "pipeline_metrics.json"

def get_latest_metrics():
    """Helper function to read the latest pipeline run metrics."""
    if not os.path.exists(LOG_FILE):
        raise HTTPException(status_code=404, detail="Metrics log not found. Pipeline hasn't run yet.")
    try:
        with open(LOG_FILE, "r") as f:
            lines = f.readlines()
            if not lines:
                raise HTTPException(status_code=404, detail="Metrics log is empty.")
            latest_log = json.loads(lines[-1].strip())
            return latest_log.get("metrics", latest_log)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error reading metrics: {str(e)}")

@app.get("/api/v1/health")
def health_check():
    """Application deployment and pipeline execution status."""
    try:
        metrics = get_latest_metrics()
        return {
            "status": "healthy",
            "deployment": "active",
            "last_pipeline_execution_status": metrics.get("execution_status", "unknown"),
            "last_execution_time_seconds": metrics.get("execution_time_seconds", 0)
        }
    except HTTPException:
        return {"status": "healthy", "deployment": "active", "pipeline_status": "waiting_for_first_run"}

@app.get("/api/v1/pipeline/flow")
def get_pipeline_flow():
    """Structural execution steps of the preprocessing and EDA pipeline."""
    return {
        "pipeline_steps": [
            {"step": 1, "name": "Data Ingestion", "description": "Download dataset from Kaggle and load into Pandas DataFrame."},
            {"step": 2, "name": "Data Preprocessing", "description": "Handle missing values via median imputation and scale numerical features using StandardScaler."},
            {"step": 3, "name": "Exploratory Data Analysis", "description": "Generate correlation matrix, bin continuous variables, and apply One-Hot Encoding."},
            {"step": 4, "name": "Feature Importance Extraction", "description": "Train a Random Forest classifier to extract Gini-based feature importances."},
            {"step": 5, "name": "DataOps & Automation", "description": "Schedule execution every 2 minutes and stream logs to centralized JSON."}
        ]
    }

@app.get("/api/v1/metrics/preprocessing")
def get_preprocessing_metrics():
    """Count of imputed missing values, record sizes, and scaling parameters."""
    metrics = get_latest_metrics()
    return {
        "records_processed": metrics.get("records_processed", 0),
        "missing_values_imputed": metrics.get("missing_values_imputed", 0),
        "scaling_method": "StandardScaler",
        "imputation_strategy": "median"
    }

@app.get("/api/v1/eda/feature-importance")
def get_feature_importance():
    """Top feature importance scores driving loan default predictions."""
    metrics = get_latest_metrics()
    return {
        "top_predictive_feature": metrics.get("top_feature_driver", "N/A"),
        "importance_score": metrics.get("top_feature_score", 0.0),
        "model_used": "RandomForestClassifier(n_estimators=50, max_depth=10)"
    }

# Conclusion for Member 4

The built-in API layer and preparation for the final deliverables are complete.

### Key Outcomes
1. **API Endpoints:** Four structured `GET` endpoints were created using FastAPI to expose health metrics, pipeline structural flow, preprocessing data, and EDA feature importance.
2. **API Testing (Task 4.2):** Once the `api_app.py` server is running, the interactive API client (Swagger UI) is available at `http://localhost:8000/docs`. This UI is used to test the endpoints and capture the JSON response objects and HTTP 200 OK status codes required for the final report.

### Task 4.3: Final Deliverables Assembly Checklist
To complete the group submission, the following must be assembled:
*   **The Report (`<groupid>.docx` or `.pdf`):** Combine the architecture details, the three EDA plots saved by Member 2, the API Swagger UI screenshots, and a clear matrix of member contributions. (Only one member uploads this to the portal).
*   **The Video Demonstration:** Record a live screen-capture video showcasing the terminal running the 2-minute pipeline scheduled tasks, the browser displaying the auto-updating Streamlit dashboard, and successful API calls made via Swagger UI. Upload this to a shared Google Drive.